# Case study: the same aggregation in SQL, pandas and Polars on the raw EBTI export

*Session 3, block 3 · Author: course team · Licence: CC-BY-4.0*

**Question.** For each issuing country and start year: how many decisions, how many different headings,
how long is the description on average (characters), and what share of decisions carries an invalidation
code (the decision ended before its normal three-year expiry, for example because the nomenclature changed
or the classification was found to be wrong)? Keep country-years with at least 1,000 decisions.

We answer it on the **official raw export**: 1,051,034 decisions of 2004–2026 in 23 yearly CSV files,
about 1.1 GB once unzipped. This is the file `prepare_data.py` downloaded to `case-study/data/raw/`. Unlike
the Parquet files of the course, a CSV file has no column types and no compression, so every tool must parse
1.1 GB of text. We compare the **code**, the **runtime** and the **memory use**.

Your timings will differ from your neighbours': they depend on the processor, the number of cores, the
disk and the library versions. Compare the *ratios*, not the absolute numbers.

In [ ]:
import os
import platform
import tempfile
import time
import zipfile
from pathlib import Path

import duckdb
import pandas as pd
import polars as pl

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
ZIP = ROOT / "case-study" / "data" / "raw" / "DDS2-EBTI_Full.zip"
assert ZIP.exists(), "run case-study/prepare_data.py first (it downloads the raw export)"

# the CSV files are extracted once, outside the repository (set EBTI_CSV_DIR to choose another folder)
CSV_DIR = Path(os.environ.get("EBTI_CSV_DIR", Path(tempfile.gettempdir()) / "ebti-csv"))
if not (CSV_DIR / "EBTI_2026.csv").exists():
    t0 = time.perf_counter()
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(CSV_DIR)
    print(f"extracted in {time.perf_counter() - t0:.0f} s")
FILES = sorted(CSV_DIR.glob("EBTI_*.csv"))
print(len(FILES), "files,", round(sum(f.stat().st_size for f in FILES) / 1e9, 2), "GB in", CSV_DIR)
print(platform.platform(), "| pandas", pd.__version__, "| polars", pl.__version__, "| duckdb", duckdb.__version__)

In [ ]:
# a first look: the columns and two rows (the file starts with a byte-order mark, hence utf-8-sig)
pd.read_csv(FILES[-1], dtype=str, encoding="utf-8-sig", nrows=2).T

Three things to notice: all values are text; the dates are written `dd/mm/yyyy`, so the year is the
last four characters of `START_DATE_OF_VALIDITY`; and `NOMENCLATURE_CODE` looks like
`6404199000********0***`, so the heading is its first four characters. One column is called `DATE_OF _ISSUE`,
with a space: raw data are rarely tidy.

## 1. SQL (DuckDB on the CSV files)

In [ ]:
SQL = f"""
SELECT ISSUING_COUNTRY AS country,
       TRY_CAST(substr(START_DATE_OF_VALIDITY, 7, 4) AS INTEGER) AS year,
       COUNT(*) AS n,
       COUNT(DISTINCT substr(NOMENCLATURE_CODE, 1, 4)) AS n_headings,
       AVG(length(DESCRIPTION_OF_GOODS)) AS avg_chars,
       AVG(CASE WHEN coalesce(INVALIDATION_REASON, '') <> '' THEN 1.0 ELSE 0.0 END) AS share_invalidated
FROM read_csv('{CSV_DIR}/EBTI_*.csv', all_varchar = true, parallel = false)
GROUP BY country, year
HAVING COUNT(*) >= 1000 AND year BETWEEN 2004 AND 2026
ORDER BY country, year
"""


def run_sql():
    return duckdb.sql(SQL).pl()        # result as a Polars DataFrame


res_sql = run_sql()
res_sql.head()

`parallel = false`: DuckDB's parallel CSV reader splits the file into chunks and cannot handle line
breaks inside quoted text, which the descriptions contain. Reading with one thread is the price.

## 2. pandas (eager)

pandas executes each line immediately (**eager evaluation**): it reads every file completely, all 15 text
columns, then builds every intermediate table in memory.

In [ ]:
def run_pandas():
    raw = pd.concat([pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in FILES], ignore_index=True)
    return (
        raw.assign(country=raw["ISSUING_COUNTRY"],
                   year=pd.to_numeric(raw["START_DATE_OF_VALIDITY"].str[6:10], errors="coerce"),
                   heading=raw["NOMENCLATURE_CODE"].str[:4],
                   n_chars=raw["DESCRIPTION_OF_GOODS"].str.len(),
                   invalidated=raw["INVALIDATION_REASON"].fillna("") != "")
        .groupby(["country", "year"], as_index=False)
        .agg(n=("heading", "size"), n_headings=("heading", "nunique"),
             avg_chars=("n_chars", "mean"), share_invalidated=("invalidated", "mean"))
        .query("n >= 1000 and 2004 <= year <= 2026")
        .sort_values(["country", "year"])
    )


res_pd = run_pandas()
res_pd.head()

## 3. Polars: eager and lazy

Polars code is built from **expressions** such as `pl.col("NOMENCLATURE_CODE").str.slice(0, 4)`. With
`scan_csv` instead of `read_csv`, nothing is computed yet: Polars builds a **query plan** (a `LazyFrame`),
optimises it and runs it on `.collect()`.

In [ ]:
def build_query(source: pl.LazyFrame) -> pl.LazyFrame:
    return (
        source.select(
            country=pl.col("ISSUING_COUNTRY"),
            year=pl.col("START_DATE_OF_VALIDITY").str.slice(6, 4).cast(pl.Int32, strict=False),
            heading=pl.col("NOMENCLATURE_CODE").str.slice(0, 4),
            n_chars=pl.col("DESCRIPTION_OF_GOODS").str.len_chars(),
            invalidated=pl.col("INVALIDATION_REASON").fill_null("") != "",
        )
        .group_by("country", "year")
        .agg(n=pl.len(), n_headings=pl.col("heading").n_unique(),
             avg_chars=pl.col("n_chars").mean(), share_invalidated=pl.col("invalidated").mean())
        .filter(pl.col("n") >= 1000, pl.col("year").is_between(2004, 2026))
        .sort("country", "year")
    )


def run_polars_eager():
    # read everything first, then compute (like pandas)
    return build_query(pl.concat([pl.read_csv(f, infer_schema=False) for f in FILES]).lazy()).collect()


def run_polars_lazy():
    return build_query(pl.scan_csv(FILES, infer_schema=False)).collect()


def run_polars_streaming():
    # the streaming engine processes the files in batches; it also works for data larger than memory
    return build_query(pl.scan_csv(FILES, infer_schema=False)).collect(engine="streaming")


res_pl = run_polars_lazy()
res_pl.head()

### What did the optimiser do?

`explain()` prints the optimised plan. Look for the line that scans the CSV files: Polars parses only the
columns it needs (**projection pushdown**: 5 of 15 columns). A filter on the input, for example
`ISSUING_COUNTRY == "DE"`, would also be pushed into the scan (**predicate pushdown**).

In [ ]:
print(build_query(pl.scan_csv(FILES, infer_schema=False)).explain())

## 4. Are the results the same?

Before comparing speed, check that the three tools answer the same question. The groups and counts must
agree exactly; averages may differ in the last digits.

In [ ]:
from polars.testing import assert_frame_equal

key = ["country", "year"]
a = res_pl.sort(key)
b = pl.from_pandas(res_pd).sort(key)
c = res_sql.sort(key)
for other in (b, c):
    assert_frame_equal(a, other, check_dtypes=False, abs_tol=1e-9)
print("all three agree:", a.shape)

This check passes only because each version says explicitly what counts as "no invalidation reason".
The three tools read an empty field differently: Polars returns an empty string `""`, pandas and DuckDB
return a missing value (`NaN`, `NULL`). The cell below shows it. A rule such as `INVALIDATION_REASON IS NOT
NULL` would have given 100 % for every country-year in Polars and plausible numbers in the other two tools:
a typical finding when the same rule is implemented twice. Decide on one definition, write it down
(Session 4) and test it.

In [ ]:
first = FILES[0]
print("Polars null count:", pl.read_csv(first, infer_schema=False)["INVALIDATION_REASON"].null_count())
print("pandas NaN count: ", pd.read_csv(first, dtype=str, encoding="utf-8-sig")["INVALIDATION_REASON"].isna().sum())
print("DuckDB NULL count:", duckdb.sql(f"SELECT COUNT(*) - COUNT(INVALIDATION_REASON) FROM read_csv('{first}', all_varchar = true, parallel = false)").fetchone()[0])

## 5. Runtime

Each version runs three times; we keep the fastest run (the others include warm-up effects such as the
operating system's file cache). The pandas version takes the longest; be patient.

In [ ]:
def best_of(fn, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


runs = {"SQL (DuckDB)": run_sql, "pandas": run_pandas, "Polars eager": run_polars_eager,
        "Polars lazy": run_polars_lazy, "Polars streaming": run_polars_streaming}
runtime = pd.Series({name: best_of(fn) for name, fn in runs.items()}, name="seconds").round(2)
runtime.to_frame().assign(relative_to_pandas=lambda d: (d["seconds"] / d.loc["pandas", "seconds"]).round(2))

## 6. Memory

**In-memory size of the table.** pandas reports the bytes of each column (`deep=True` includes the
strings); Polars estimates the size of its Arrow buffers. Both hold all 15 columns of all 1,051,034 rows.

In [ ]:
raw_pd = pd.concat([pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in FILES], ignore_index=True)
raw_pl = pl.concat([pl.read_csv(f, infer_schema=False) for f in FILES])
print(f"pandas: {raw_pd.memory_usage(deep=True).sum() / 1e9:,.2f} GB")
print(f"Polars: {raw_pl.estimated_size() / 1e9:,.2f} GB")
print(f"CSV files on disk: {sum(f.stat().st_size for f in FILES) / 1e9:,.2f} GB; zip: {ZIP.stat().st_size / 1e9:,.2f} GB")
del raw_pd, raw_pl

**Peak memory of the whole computation.** Memory that is freed again does not show up in the table size,
so we measure the **peak resident memory** of a fresh Python process for each version. The baseline is a
process that only imports the libraries.

In [ ]:
import subprocess
import sys
import textwrap

BENCH = textwrap.dedent(f"""
    import resource, sys
    import duckdb, pandas as pd, polars as pl
    FILES = {[str(f) for f in FILES]!r}
    version = sys.argv[1]
    if version == "pandas":
        raw = pd.concat([pd.read_csv(f, dtype=str, encoding="utf-8-sig") for f in FILES], ignore_index=True)
        out = (raw.assign(year=raw["START_DATE_OF_VALIDITY"].str[6:10], heading=raw["NOMENCLATURE_CODE"].str[:4],
                          n_chars=raw["DESCRIPTION_OF_GOODS"].str.len(),
                          invalidated=raw["INVALIDATION_REASON"].fillna("") != "")
                  .groupby(["ISSUING_COUNTRY", "year"])
                  .agg(n=("heading", "size"), n_headings=("heading", "nunique"),
                       avg_chars=("n_chars", "mean"), share_invalidated=("invalidated", "mean")))
    elif version.startswith("polars"):
        src = pl.concat([pl.read_csv(f, infer_schema=False) for f in FILES]).lazy() if version == "polars_eager" else pl.scan_csv(FILES, infer_schema=False)
        q = (src.select(country=pl.col("ISSUING_COUNTRY"), year=pl.col("START_DATE_OF_VALIDITY").str.slice(6, 4),
                        heading=pl.col("NOMENCLATURE_CODE").str.slice(0, 4),
                        n_chars=pl.col("DESCRIPTION_OF_GOODS").str.len_chars(),
                        invalidated=pl.col("INVALIDATION_REASON").fill_null("") != "")
                .group_by("country", "year")
                .agg(n=pl.len(), n_headings=pl.col("heading").n_unique(),
                     avg_chars=pl.col("n_chars").mean(), share_invalidated=pl.col("invalidated").mean()))
        out = q.collect(engine="streaming" if version == "polars_streaming" else "auto")
    elif version == "duckdb":
        out = duckdb.sql(sys.stdin.read()).pl()
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(peak / (1e6 if sys.platform == "darwin" else 1e3))   # bytes on macOS, kB on Linux -> MB
""")


def peak_mb(version: str) -> float:
    result = subprocess.run([sys.executable, "-c", BENCH, version], input=SQL,
                            capture_output=True, text=True, check=True)
    return float(result.stdout.strip().splitlines()[-1])


versions = {"baseline (imports only)": "baseline", "SQL (DuckDB)": "duckdb", "pandas": "pandas",
            "Polars eager": "polars_eager", "Polars lazy": "polars_lazy", "Polars streaming": "polars_streaming"}
peak = pd.Series({name: peak_mb(v) for name, v in versions.items()}, name="peak_MB").round(0)
peak.to_frame().assign(above_baseline=lambda d: d["peak_MB"] - d.loc["baseline (imports only)", "peak_MB"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
runtime.plot.barh(ax=axes[0], color="#2a78d6")
axes[0].set(title="Runtime (best of 3)", xlabel="seconds")
(peak.drop("baseline (imports only)") - peak["baseline (imports only)"]).plot.barh(ax=axes[1], color="#eb6834")
axes[1].set(title="Peak memory above baseline", xlabel="MB")
for ax in axes:
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

**How to read the memory numbers.** When the course team ran this notebook (Apple silicon laptop,
pandas 2.3, Polars 1.44, DuckDB 1.5), pandas needed about 9.5 s and Polars lazy about 0.2 s: a factor of
about 50. The peak memory, however, was similar for pandas and all Polars versions (about 2.1 to 2.4 GB
above the baseline); DuckDB needed about 1.3 GB. The lazy and streaming versions did *not* save memory here.
One likely reason: Polars maps the CSV files into memory to parse them in parallel, and mapped file pages
count as resident memory although the operating system can release them at any time. Peak resident memory
is therefore an upper bound, not the memory the computation really needs. Check your own numbers before you
repeat claims such as "lazy always saves memory".

## 7. Compare the code

| Step | SQL | pandas | Polars |
|---|---|---|---|
| read | `FROM read_csv(...)` | `pd.read_csv` per file, `pd.concat` (eager) | `pl.scan_csv(files)` (lazy) |
| new column | expression in `SELECT` | `assign(...)` | expression in `select` |
| empty text field | `NULL` | `NaN` | `""` |
| group | `GROUP BY` | `groupby` | `group_by` |
| distinct count | `COUNT(DISTINCT ...)` | `nunique` | `n_unique` |
| filter groups | `HAVING` | `query` after `agg` | `filter` after `agg` |

**Questions for discussion**

1. Which version was fastest on your machine, and by what factor? Which used the least memory?
2. pandas can also read only the columns it needs: `pd.read_csv(f, usecols=[...])`. How much does that
   change its runtime and memory use? Why can it not match the lazy Polars version completely?
3. Add a filter `ISSUING_COUNTRY == "DE"` *before* the `select` in the Polars query. Where does it appear in
   `explain()`?
4. The raw export contains three decisions with a start year after 2026 (2027, 2055, 2200). Which part of
   each query removes them? Is removing them the right decision?
5. Your project data have 10 million rows and live in a PostgreSQL database shared by the team. Which tool
   would you use for which step, and why?

In [ ]:
# space for questions 2-4